In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import mlflow
import mlflow.sklearn
import os


# Fijar directorio raíz del proyecto
os.chdir(r"c:\Users\emerd\Desktop\repos\churn-prediction")
print("Directorio:", os.getcwd())

# Imprimir versiones para debugging
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", __import__('sklearn').__version__)
print("MLflow:", mlflow.__version__)

# Configurar MLflow al inicio (ruta relativa para portabilidad)
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("churn-prediction")

# Cargar datos
df = pd.read_csv('data/Churn_Modelling.csv')

# Preprocessing
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])
df = pd.get_dummies(df, columns=['Geography'], drop_first=True)
# Split
X = df.drop('Exited', axis=1)
y = df['Exited']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Escalar
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Todo listo ✓")

Directorio: c:\Users\emerd\Desktop\repos\churn-prediction
Pandas: 2.3.3
NumPy: 2.4.4
Scikit-learn: 1.8.0
MLflow: 3.11.1


2026/05/05 00:33:50 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/05 00:33:50 INFO mlflow.store.db.utils: Updating database tables
2026/05/05 00:33:53 INFO mlflow.tracking.fluent: Experiment with name 'churn-prediction' does not exist. Creating a new experiment.


Todo listo ✓


---

## Fine Tuning

In [2]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs= 1, #-1 seria ideal pero no tengo mucha ram
    verbose=1
)

grid_search.fit(X_train_scaled, y_train)

print("Mejores hiperparámetros:", grid_search.best_params_)
print("Mejor AUC-ROC en CV:", grid_search.best_score_.round(3))

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Mejores hiperparámetros: {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 300}
Mejor AUC-ROC en CV: 0.862


In [3]:
best_model = grid_search.best_estimator_
print(best_model)
y_pred_best = best_model.predict(X_test_scaled)
y_proba_best = best_model.predict_proba(X_test_scaled)[:,1]

print("=== Random Forest Optimizado ===")
print(classification_report(y_test, y_pred_best))
print("AUC-ROC:", roc_auc_score(y_test, y_proba_best))

RandomForestClassifier(max_depth=10, n_estimators=300, random_state=42)
=== Random Forest Optimizado ===
              precision    recall  f1-score   support

           0       0.87      0.97      0.92      1593
           1       0.82      0.45      0.58       407

    accuracy                           0.87      2000
   macro avg       0.85      0.71      0.75      2000
weighted avg       0.86      0.87      0.85      2000

AUC-ROC: 0.8640875081553048


> Precision Vista Modelo y 
> Recall Vista Real Negocio

In [4]:
with mlflow.start_run():
    mlflow.log_params(grid_search.best_params_)
    
    auc = roc_auc_score(y_test, y_proba_best)
    report = classification_report(y_test, y_pred_best, output_dict=True)
    
    mlflow.log_metric("auc_roc", auc)
    mlflow.log_metric("recall_churn", report['1']['recall'])
    mlflow.log_metric("precision_churn", report['1']['precision'])
    mlflow.log_metric("f1_churn", report['1']['f1-score'])
    
    mlflow.sklearn.log_model(best_model, "random_forest_churn")
    
    print("Experimento guardado en MLflow ✓")

2026/05/05 00:37:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/05 00:37:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Experimento guardado en MLflow ✓


In [5]:
client = mlflow.tracking.MlflowClient()
experiments = client.search_experiments()
for exp in experiments:
    print(exp.name, exp.experiment_id)

churn-prediction 1
Default 0


In [6]:
runs = client.search_runs(experiment_ids=["1"])
for run in runs:
    print("Run ID:", run.info.run_id)
    print("Métricas:", run.data.metrics)
    print("Parámetros:", run.data.params)

Run ID: e58fa659c0c14ce899b8907127b51110
Métricas: {'auc_roc': 0.8640875081553048, 'recall_churn': 0.44717444717444715, 'precision_churn': 0.8198198198198198, 'f1_churn': 0.5786963434022258}
Parámetros: {'max_depth': '10', 'min_samples_split': '2', 'n_estimators': '300'}


In [7]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

experiments = client.search_experiments()
for exp in experiments:
    print(exp.experiment_id, exp.name)

runs = client.search_runs(experiment_ids=["1"])
print(len(runs))

1 churn-prediction
0 Default
1
